# **Gradient-free CAM: Score-CAM, Ablation-CAM, Eigen-CAM**

Practice for the module [«Attribution: localisation and the CAM family»](https://ai-interpretability.school).

The lesson took apart three ways of getting the weights of feature maps without asking the
gradient, and said what each of them pays. Here all of it is computed by hand — on top of the
same ResNet-50 as in the Grad-CAM practice.

By the end of the notebook you will have:

- your own Score-CAM, Ablation-CAM and Eigen-CAM (about ten lines each);
- the measured price of giving up the gradient — in forward passes, not in words;
- a check that Eigen-CAM **does not depend on the class**, on an image with two objects.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

## The model, the image and the shared part

Every method of the family is built the same way: the map is a weighted sum of feature maps
$\sum_k w^c_k A^k$, and they differ only in where the weights come from. So the shared part is
getting the feature maps of the last convolutional layer.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# There are two objects of different classes in this photograph — we need it at the end.
image = Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg').content)).convert('RGB')
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    logits = model(x)
top = torch.topk(logits, 3).indices[0].tolist()
print('top-3:', [(i, categories[i]) for i in top])

In [ ]:
def feature_maps(model, x, layer):
    """Feature maps of the given layer: a tensor (C, h, w)."""
    store = {}
    handle = layer.register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    with torch.no_grad():
        model(x)
    handle.remove()
    return store['a'][0]

target_layer = model.layer4[-1]
A = feature_maps(model, x, target_layer)
print('feature maps:', A.shape[0], '· size of each:', tuple(A.shape[1:]))

## 1. Score-CAM: ask the model directly

The weight of a channel is the logit of the target class on the image masked by that channel.
No backward pass: forward only.

$$w^c_k = f_c(x \odot \text{norm}(A^k)) - f_c(\text{blank image})$$

In [ ]:
@torch.no_grad()
def score_cam(model, x, layer, cls, batch=64):
    A = feature_maps(model, x, layer)
    C = A.shape[0]

    # upsample the maps to the input size and normalize to [0, 1] — these are the masks
    M = F.interpolate(A.unsqueeze(0), x.shape[-2:], mode='bilinear', align_corners=False)[0]
    lo = M.flatten(1).min(1).values[:, None, None]
    hi = M.flatten(1).max(1).values[:, None, None]
    M = (M - lo) / (hi - lo + 1e-8)

    base = model(torch.zeros_like(x))[0, cls]      # the logit on a blank image
    weights, passes = [], 1
    for i in range(0, C, batch):
        masked = x * M[i:i + batch].unsqueeze(1)
        weights.append(model(masked)[:, cls] - base)
        passes += masked.shape[0]
    w = torch.cat(weights)

    cam = torch.relu((w[:, None, None] * A).sum(0))
    return cam / (cam.max() + 1e-8), passes

cls = top[0]
sc_cam, sc_passes = score_cam(model, x, target_layer, cls)
print(f'Score-CAM: forward passes {sc_passes}')

**Task 1.** How many forward passes would Score-CAM need on `layer3` instead of `layer4`?
Work it out without running: the number of channels of the layer is visible in `print(model)`.

In [ ]:
# Your code here

## 2. Ablation-CAM: switch it off and look

The mirror idea: the weight of a channel is how much the logit drops if the channel is zeroed.

$$w^c_k = \frac{y^c - y^c_{\setminus k}}{y^c}$$

There is no need to recompute the whole network — the tail after the layer of interest is enough.
Here, for clarity, we zero the channel right in the feature maps and run the rest of the network.

In [ ]:
@torch.no_grad()
def ablation_cam(model, x, layer, cls):
    A = feature_maps(model, x, layer)
    C = A.shape[0]

    def head(maps):
        """The ResNet tail after layer4: pooling and the classifier."""
        v = model.avgpool(maps.unsqueeze(0)).flatten(1)
        return model.fc(v)[0, cls]

    y = head(A)
    weights = []
    for k in range(C):
        ablated = A.clone()
        ablated[k] = 0
        weights.append((y - head(ablated)) / (y + 1e-8))
    w = torch.stack(weights)

    cam = torch.relu((w[:, None, None] * A).sum(0))
    return cam / (cam.max() + 1e-8)

ab_cam = ablation_cam(model, x, target_layer, cls)
print('Ablation-CAM computed')

**Task 2.** The lesson says: when several channels duplicate each other, Ablation-CAM
underrates each of them — switch one off and the neighbours compensate. Find the share of
channels whose weight is smaller than 0.001 in absolute value. What kind of channels are those?

In [ ]:
# Your code here

## 3. Eigen-CAM: no class at all

We reshape the activations into a «positions × channels» matrix and take the first principal
component. The class does not take part at all — and that is the key property of the method.

In [ ]:
@torch.no_grad()
def eigen_cam(model, x, layer):
    A = feature_maps(model, x, layer)
    C, h, w = A.shape
    flat = A.reshape(C, h * w).T                    # (positions, channels)
    flat = flat - flat.mean(0, keepdim=True)
    _, _, V = torch.linalg.svd(flat, full_matrices=False)
    cam = (flat @ V[0]).reshape(h, w)               # projection onto the first component
    cam = torch.relu(cam)
    return cam / (cam.max() + 1e-8)

ei_cam = eigen_cam(model, x, target_layer)
print('Eigen-CAM computed')

## 4. The main check: does the map depend on the class

There are two objects in the photograph. We build maps for two different classes and see whether
anything has changed. A method honest to the class must give maps that diverge.

In [ ]:
# Two classes of different nature: the dog one from top-1 and a cat one — the photo has both.
cls_a = top[0]
cls_b = max(range(len(categories)),
            key=lambda i: logits[0, i].item() if 'cat' in categories[i] else -1e9)
print('class A:', categories[cls_a], '· class B:', categories[cls_b])

def diff(m1, m2):
    """Mean difference between two maps normalized to [0, 1]."""
    return (m1 - m2).abs().mean().item()

sc_a, _ = score_cam(model, x, target_layer, cls_a)
sc_b, _ = score_cam(model, x, target_layer, cls_b)
ab_a = ablation_cam(model, x, target_layer, cls_a)
ab_b = ablation_cam(model, x, target_layer, cls_b)
ei = eigen_cam(model, x, target_layer)

print(f'Score-CAM    A against B: {diff(sc_a, sc_b):.4f}')
print(f'Ablation-CAM A against B: {diff(ab_a, ab_b):.4f}')
print(f'Eigen-CAM    A against B: {diff(ei, ei):.4f}   ← the class does not enter the method at all')

**Task 3.** What is the mean difference between the Score-CAM maps for the two classes
(round to hundredths)? And for Eigen-CAM — and why exactly that much?

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(image.resize((224, 224))); ax[0].set_title('original')
for a, (m, t) in zip(ax[1:], [(sc_a, 'Score-CAM'), (ab_a, 'Ablation-CAM'), (ei, 'Eigen-CAM')]):
    a.imshow(image.resize((224, 224)))
    a.imshow(F.interpolate(m[None, None], (224, 224), mode='bilinear')[0, 0].numpy(),
             cmap='jet', alpha=0.5)
    a.set_title(t)
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()

## What to take away

| Method | What it pays | When to take it |
| --- | --- | --- |
| Score-CAM | a forward pass per channel | the gradients are noisy and you have time |
| Ablation-CAM | the same, but only the tail is recomputed | you need importance «in context» of the other channels |
| Eigen-CAM | **does not depend on the class** | quick debugging, tasks without logits |

And what does not change when you change the way the weights are obtained: all three methods live
in the resolution of the feature maps ($7\times7$ for a $224\times224$ input) and answer the
question «in which region», not «at which pixels».